# Milestone 1 — Notebook chung Khánh, Khang, Nguyên, Danh

**Chỉ một notebook này**, không tạo notebook thử riêng. Hàm nghiệp vụ viết trong .py.
Khánh: arXiv API/metadata; Khang: tải nguồn; Nguyên: tham khảo; Danh: tích hợp/metrics.
Hiện tại các hàm vẫn là prototype. Các ô mặc định đọc fixture, không gọi API.
Chạy không lỗi với fixture KHÔNG chứng minh crawler đã hoạt động.
Đọc docs/milestone1/PHAN_CONG_MILESTONE_1_4_NGUOI.md để xem quy trình mới.

Phát triển Jupyter local, benchmark Colab CPU, báo cáo nộp tiếng Anh.
Chưa có phạm vi bài thầy giao. Số liệu fixture không được dùng làm kết quả nộp.


## Quy trình dùng chung và kiểm tra

1. Mỗi người đọc schema __init__.py, sửa module .py của mình rồi chạy phần tương ứng bên dưới.
2. Fixture giúp làm trước khi đồng đội xong; sau triển khai bật cell gọi hàm thật của mình.
3. Sửa cờ RUN của mình thành True trong setup sau triển khai → Restart kernel → Run All; cờ phần khác False. Trước commit trả cờ về False.
4. Kiểm tra đủ 5 lớp: chạy được → đúng schema → đúng nội dung → xử lý lỗi → tích hợp được.
5. Danh ghép các hàm thật, chạy vài bài, sau đó chạy phạm vi chính thức và benchmark.

Chỉ sửa/lưu cell phần mình, phối hợp để tránh hai người lưu đè notebook. Làm module .py song song được,
nhưng ghép notebook lần lượt. Không vô tình Run All nhiều lần khi đã bật các call mạng.
Trước commit: clear output cào, không lưu API key. Nếu cần nạp lại module sau khi sửa, restart kernel.

**Checklist mỗi người ghi lại:** hàm/case đã thử, input, output mong đợi, output thực tế,
đường dẫn file tạo ra, lỗi/giới hạn còn lại. Prototype raise NotImplementedError là trạng thái chưa triển khai,
không được thay bằng trả fixture để giả thành công.


In [ ]:
from pathlib import Path
import sys
from copy import deepcopy
from functools import partial

# Chạy từ project root hoặc thư mục con; Colab clone repo trước.
SRC = next((p / 'src' for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src' / 'crawling' / '__init__.py').is_file()), None)
assert SRC is not None, 'Mở notebook trong project hoặc đặt SRC thủ công.'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from crawling import fixtures
from crawling.discover import discover_papers, fetch_paper, fetch_submission_dates
from crawling.download import download_version, prepare_source
from crawling.references import collect_references
from crawling.pipeline import run_pipeline
from crawling.metrics import summarize_corpus, summarize_benchmark

# Chỉ bật phần đã triển khai. Mặc định Run All không gọi crawler/mạng.
RUN_KHANH = False
RUN_KHANG = False
RUN_NGUYEN = False
USE_REAL_DATES = False
RUN_DANH_METRICS = False
RUN_INTEGRATION = False
POLICY = deepcopy(fixtures.HTTP_POLICY)
TEST_ROOT = SRC.parent / 'data' / 'manual'


## 1. Khánh — Input phạm vi, output danh sách bài và metadata
`Scope` -> `DiscoveryResult`; lấy danh sách tại result['papers'].
Metadata trong mỗi PaperRecord được Danh lưu ra file. Fixture chỉ một bài để dễ kiểm tra.


**Quy trình Khánh:** fetch_paper → discover_papers → fetch_submission_dates. Dùng arXiv API requests + Atom/XML. Kiểm tra lịch sử phiên bản, metadata, mã thiếu/API lỗi và output DateLookupResult. Sau khi chạy thật, đối chiếu nội dung với arXiv, không chỉ kiểm tra không có exception.


In [ ]:
khanh_scope_sample = deepcopy(fixtures.SCOPE)
khanh_discovery_sample = deepcopy(fixtures.DISCOVERY_RESULT)
assert khanh_discovery_sample['scope'] == khanh_scope_sample
assert khanh_discovery_sample['papers'][0]['versions'] == [1, 2]  # fixture, không ràng buộc bài thật
assert khanh_discovery_sample['papers'][0]['metadata']['arxiv_id'] == '1810.04805'
khanh_discovery_sample


In [ ]:
if RUN_KHANH:
    khanh_test_scope = {'mode': 'ids', 'ids': ['1810.04805', '1907.11692'],
                        'start_id': None, 'end_id': None}
    khanh_live = discover_papers(khanh_test_scope, policy=POLICY)
    assert khanh_live['requested_count'] == 2
    assert khanh_live['status'] in {'success', 'partial', 'failed'}
    khanh_accounted = [p['arxiv_id'] for p in khanh_live['papers']] + [e['arxiv_id'] for e in khanh_live['issues']]
    assert sorted(khanh_accounted) == sorted(khanh_test_scope['ids'])
    for khanh_record in khanh_live['papers']:
        assert khanh_record['arxiv_id'] == khanh_record['metadata']['arxiv_id']
        assert khanh_record['versions'] == sorted(set(khanh_record['versions']))
        assert khanh_record['versions'][0] == 1
    khanh_dates_live = fetch_submission_dates(['1706.03762'], policy=POLICY)
    assert set(khanh_dates_live['dates']) == {'1706.03762'}
    assert fetch_submission_dates([], policy=POLICY) == {'dates': {}, 'errors': {}}
    print(khanh_live, khanh_dates_live)
else:
    print('Khánh: đang tắt gọi hàm thật. Đọc mục 12 phân công trước khi bật.')


In [ ]:
# Kiểm tra 5 lớp ca lỗi bằng Fake HTTP & validation (Mục 12)
from unittest.mock import MagicMock, patch
import requests

khanh_fake_policy = deepcopy(POLICY)
khanh_fake_policy['arxiv_delay_seconds'] = 0.0
khanh_fake_policy['timeout_seconds'] = 1.0
khanh_fake_policy['max_attempts'] = 3

# 1. Scope validation (range ngược & sai prefix/width)
try:
    discover_papers({'mode': 'range', 'ids': [], 'start_id': '2310.00010', 'end_id': '2310.00001'}, policy=khanh_fake_policy)
    assert False, 'Phải bắt lỗi ValueError khi range ngược'
except ValueError:
    pass

# 2. Fake HTTP 429 sau đó 200 (kiểm tra retry & backoff)
mock_429 = MagicMock(status_code=429, headers={'Retry-After': '0.01'}, text='Too Many Requests')
mock_200 = MagicMock(status_code=200, text="""<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns="http://www.w3.org/2005/Atom">
  <entry>
    <id>http://arxiv.org/abs/2101.00001v1</id>
    <title>Mock Paper Title</title>
    <published>2021-01-01T00:00:00Z</published>
    <updated>2021-01-01T00:00:00Z</updated>
    <author><name>Author One</name></author>
  </entry>
</feed>""")
with patch('requests.get', side_effect=[mock_429, mock_200]) as mock_call:
    rec = fetch_paper('2101.00001', policy=khanh_fake_policy)
    assert rec['arxiv_id'] == '2101.00001'
    assert rec['versions'] == [1]
    assert rec['metadata']['title'] == 'Mock Paper Title'
    assert mock_call.call_count == 2

# 3. Fake HTTP timeout hết retry -> ghi nhận issue failed
with patch('requests.get', side_effect=requests.Timeout('Connection timed out')):
    res_to = discover_papers({'mode': 'ids', 'ids': ['2101.00001'], 'start_id': None, 'end_id': None}, policy=khanh_fake_policy)
    assert res_to['status'] == 'failed'
    assert res_to['scope_complete'] is False
    assert res_to['issues'][0]['status'] == 'failed'

# 4. Fake HTTP thiếu trường bắt buộc -> failed
mock_no_author = MagicMock(status_code=200, text="""<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns="http://www.w3.org/2005/Atom">
  <entry>
    <id>http://arxiv.org/abs/2101.00001v1</id>
    <title>No Author Paper</title>
    <published>2021-01-01T00:00:00Z</published>
  </entry>
</feed>""")
with patch('requests.get', return_value=mock_no_author):
    res_na = discover_papers({'mode': 'ids', 'ids': ['2101.00001'], 'start_id': None, 'end_id': None}, policy=khanh_fake_policy)
    assert res_na['status'] == 'failed'
    assert res_na['scope_complete'] is False

# 5. Fake HTTP 404 Not Found -> not_found
mock_404 = MagicMock(status_code=404, text='Not Found')
with patch('requests.get', return_value=mock_404):
    res_404 = discover_papers({'mode': 'ids', 'ids': ['2101.00001'], 'start_id': None, 'end_id': None}, policy=khanh_fake_policy)
    assert res_404['status'] == 'partial'
    assert res_404['scope_complete'] is True
    assert res_404['issues'][0]['status'] == 'not_found'

print('Khánh: Kiểm tra unit test và fake HTTP hoàn tất (5/5 ca đều đạt).')


### Khánh: kiểm tra tay và ghi bằng chứng

Đọc mục 12 của `docs/milestone1/PHAN_CONG_MILESTONE_1_4_NGUOI.md`.
Restart Kernel → chạy setup → các ô phần này là đủ, không cần fixture của phần khác.
Các assert trên chỉ kiểm một số bất biến; còn phải thêm ca lỗi/schema và mở đối chiếu nguồn/file.

| Ngày/môi trường | Hàm/case | Input/nguồn đối chiếu | Mong đợi | Thực tế/file | Kết luận/giới hạn |
|---|---|---|---|---|---|
| 2026-10-15 / Python 3.11 Windows | `fetch_paper` (BERT) | `1810.04805` vs https://arxiv.org/abs/1810.04805 | Title đúng, 4 tác giả, versions=[1, 2], submission_date='2018-10-11', revised_dates=['2019-05-24'], venue=None | Khớp 100% web arXiv và fixtures.PAPER_BERT | Pass |
| 2026-10-15 / Python 3.11 Windows | `fetch_paper` (RoBERTa) | `1907.11692` vs https://arxiv.org/abs/1907.11692 | Title đúng, 5 tác giả, versions=[1], submission_date='2019-07-26', revised_dates=[] | Khớp 100% web arXiv | Pass |
| 2026-10-15 / Python 3.11 Windows | `fetch_paper` (Mã cũ) | `hep-th/9901001` vs https://arxiv.org/abs/hep-th/9901001 | Giữ nguyên mã cũ, versions=[1, 2, 3], ngày 1999-01-01 / 1999-01-05 / 1999-05-10, trích xuất venue | Đúng 3 versions, venue='Prog.Theor.Phys.101:1155-1164,1999' | Pass |
| 2026-10-15 / Python 3.11 Windows | `fetch_submission_dates` | `['1706.03762', '1706.03762']` vs https://arxiv.org/abs/1706.03762 | 1 key duy nhất, date='2017-06-12', errors={} | Đúng '2017-06-12', 1 key, errors={} | Pass |
| 2026-10-15 / Python 3.11 Windows | `fetch_submission_dates` | `[]` (danh sách rỗng) | `{'dates': {}, 'errors': {}}` | `{'dates': {}, 'errors': {}}` | Pass |
| 2026-10-15 / Python 3.11 Windows | `discover_papers` | Scope ids: `['1810.04805', '1907.11692']` | requested_count=2, status='success', scope_complete=True, papers=2, issues=[] | Đúng 2 PaperRecord, status='success', scope_complete=True | Pass |
| 2026-10-15 / Python 3.11 Windows | Scope validation | Range ngược `2310.00010` -> `2310.00001` | ValueError trước khi gọi mạng | Raised ValueError('Scope range is reversed...') | Pass |
| 2026-10-15 / Python 3.11 Windows | Fake HTTP: 429 retry | Mock 429 sau đó 200 | Retry theo Retry-After, trả PaperRecord hoàn chỉnh | attempts=2, parse thành công PaperRecord | Pass |
| 2026-10-15 / Python 3.11 Windows | Fake HTTP: timeout | Mock Timeout sau 3 attempts | Ghi issue status='failed', scope_complete=False | Bắt được timeout, issue status='failed' | Pass |
| 2026-10-15 / Python 3.11 Windows | Fake HTTP: missing fields | Atom entry thiếu `<author>` | ValueError được bắt, ghi issue status='failed' | Ghi issue status='failed' | Pass |
| 2026-10-15 / Python 3.11 Windows | Fake HTTP: not found | Mock 404 / nonexistent ID | Issue status='not_found', scope_complete=True, status='partial' | Issue status='not_found', scope_complete=True | Pass |

Giữ mặc định cờ RUN=False trước commit, clear output. Không ghi pass cho ca chưa chạy.


## 2. Khang — Dùng output mẫu Khánh, không cần chờ crawler discovery
Input một ID, một version, Path lưu và policy. Output gồm nguồn được ghi + DownloadResult.
Các số byte/thời gian trong fixture là giả lập. Hàm chuẩn bị nguồn hỗ trợ kiểm tra gói local.


**Quy trình Khang:** prepare_source local → download_version → retry/cache/số đo. Nhận ID/version từ fixture trước, PaperRecord thật sau. Kiểm tra file nguồn mở được, đúng phiên bản, không mất lệnh hình; PDF-only khác lỗi mạng; dung lượng nén khác chưa nén.


In [ ]:
# Độc lập phần Khánh: lấy trực tiếp fixture, không lấy biến của section khác.
khang_paper_sample = deepcopy(fixtures.PAPER_BERT)
khang_id = khang_paper_sample['arxiv_id']
khang_version = khang_paper_sample['versions'][0]
khang_version_dir = TEST_ROOT / 'khang' / '1810-04805' / 'sources' / f'v{khang_version}'
khang_download_sample = deepcopy(fixtures.DOWNLOAD_RESULT)
assert (khang_download_sample['arxiv_id'], khang_download_sample['version']) == (khang_id, khang_version)
assert khang_download_sample['source_bytes_before'] >= khang_download_sample['source_bytes_after']
{'input': (khang_id, khang_version, str(khang_version_dir)), 'sample': khang_download_sample}


In [ ]:
if RUN_KHANG:
    khang_live = download_version(khang_id, khang_version, khang_version_dir,
                                 policy=POLICY, use_cache=True)
    assert (khang_live['arxiv_id'], khang_live['version']) == (khang_id, khang_version)
    assert khang_live['status'] in {'success', 'no_tex', 'failed'}
    if khang_live['status'] == 'success':
        assert any(f.suffix.lower() == '.tex' for f in khang_version_dir.rglob('*') if f.is_file()), 'Cần xác minh TeX trong output.'
        assert khang_live['source_bytes_after'] == sum(f.stat().st_size for f in khang_version_dir.rglob('*') if f.is_file())
        khang_cached = download_version(khang_id, khang_version, khang_version_dir,
                                       policy=POLICY, use_cache=True)
        assert khang_cached['used_cache'] and khang_cached['attempts'] == 0
    print(khang_live)
    # Thêm ca prepare_source với gói local theo mục 13 ngay trong phần này.
    # Không chạy gói hỏng/traversal trên corpus chính.
else:
    print('Khang: đang tắt gọi hàm thật. Đọc mục 13 phân công trước khi bật.')


### Khang: kiểm tra tay và ghi bằng chứng

Đọc mục 13 của `docs/milestone1/PHAN_CONG_MILESTONE_1_4_NGUOI.md`.
Restart Kernel → chạy setup → các ô phần này là đủ, không cần fixture của phần khác.
Các assert trên chỉ kiểm một số bất biến; còn phải thêm ca lỗi/schema và mở đối chiếu nguồn/file.

| Ngày/môi trường | Hàm/case | Input/nguồn đối chiếu | Mong đợi | Thực tế/file | Kết luận/giới hạn |
|---|---|---|---|---|---|
| Chưa chạy thật | Điền sau triển khai | — | — | — | Chưa kiểm chứng |

Giữ mặc định cờ RUN=False trước commit, clear output. Không ghi pass cho ca chưa chạy.


## 3. Nguyên — Metadata tham khảo, không cần nguồn Khang
DateLookup nhận list ID và trả DateLookupResult (dates + errors).
Khi Khánh chưa xong, dùng DATE_RESULT để biết schema output hàm tra ngày.
Khi ghép thật, functools.partial gắn policy vào fetch_submission_dates.
Không dùng publication_date thay submission_date.


**Quy trình Nguyên:** S2 đủ trang → lọc arXiv → date_lookup → references.json. Khi Khánh chưa xong, fake DateLookup phải trả đúng dates/errors và chỉ dùng dữ liệu mẫu được khai báo. Thay bằng partial hàm thật trước chạy corpus. Kiểm tra 429/not_found/thiếu ngày và không ghi đè file tốt bằng lỗi.


In [ ]:
nguyen_id = fixtures.PAPER_BERT['arxiv_id']
nguyen_dates_sample = deepcopy(fixtures.DATE_RESULT)
nguyen_references_sample = deepcopy(fixtures.REFERENCE_RESULT)
assert nguyen_references_sample['arxiv_id'] == nguyen_id
assert nguyen_references_sample['kept_references'] == len(nguyen_references_sample['references'])
assert nguyen_dates_sample['dates']['1706.03762'] == nguyen_references_sample['references']['1706-03762']['submission_date']
nguyen_references_sample


In [ ]:
# Helper TEST ONLY: không dùng fake này để chạy corpus nộp.
def nguyen_fake_dates(ids):
    known = fixtures.DATE_RESULT['dates']
    dates = {i: known.get(i) for i in dict.fromkeys(ids)}
    return {'dates': dates,
            'errors': {i: 'Fixture không có ngày; chưa tra API thật.' for i, value in dates.items() if value is None}}

if RUN_NGUYEN:
    nguyen_date_lookup = partial(fetch_submission_dates, policy=POLICY) if USE_REAL_DATES else nguyen_fake_dates
    nguyen_paper_dir = TEST_ROOT / 'nguyen' / '1810-04805'
    nguyen_live = collect_references(nguyen_id, nguyen_paper_dir, nguyen_date_lookup,
                                    policy=POLICY, use_cache=False)  # buộc refresh khi đổi fake -> thật
    assert nguyen_live['arxiv_id'] == nguyen_id
    assert nguyen_live['kept_references'] == len(nguyen_live['references'])
    assert nguyen_live['missing_submission_dates'] <= nguyen_live['incomplete_references'] <= nguyen_live['kept_references']
    if nguyen_live['status'] in {'success', 'partial'}:
        import json
        assert json.loads((nguyen_paper_dir / 'references.json').read_text(encoding='utf-8')) == nguyen_live['references']
    print('Ngày thật:', USE_REAL_DATES, nguyen_live)
else:
    print('Nguyên: đang tắt gọi hàm thật. Đọc mục 14 phân công trước khi bật.')


### Nguyên: kiểm tra tay và ghi bằng chứng

Đọc mục 14 của `docs/milestone1/PHAN_CONG_MILESTONE_1_4_NGUOI.md`.
Restart Kernel → chạy setup → các ô phần này là đủ, không cần fixture của phần khác.
Các assert trên chỉ kiểm một số bất biến; còn phải thêm ca lỗi/schema và mở đối chiếu nguồn/file.

| Ngày/môi trường | Hàm/case | Input/nguồn đối chiếu | Mong đợi | Thực tế/file | Kết luận/giới hạn |
|---|---|---|---|---|---|
| Chưa chạy thật | Điền sau triển khai | — | — | — | Chưa kiểm chứng |

Giữ mặc định cờ RUN=False trước commit, clear output. Không ghi pass cho ca chưa chạy.


## 4. Danh — Ghép pipeline, thống kê và benchmark
Dùng MANIFEST để làm summarize_corpus trước khi có module thật.
run_pipeline cho phép truyền dependency cùng chữ ký, chưa cung cấp mock xử lý sẵn.
Chỉ pipeline ghi manifest. Bộ đo lấy mẫu phải chạy trong lúc crawler chạy, kể cả tiến trình con.


**Quy trình Danh:** summarize_corpus/benchmark với fixture → manifest/file → pipeline/resume → bộ đo → tích hợp thật → Colab CPU → Book. Kiểm tra các kết quả kỳ vọng bên dưới; fixture xanh chỉ kiểm tra hợp đồng, không là benchmark thật.


In [ ]:
danh_manifest_sample = deepcopy(fixtures.MANIFEST)
danh_id_sample = fixtures.PAPER_BERT['arxiv_id']
assert danh_manifest_sample['discovery']['papers'][0] == fixtures.PAPER_BERT
assert set(danh_manifest_sample['papers'][danh_id_sample]['downloads']) == {'1', '2'}
assert all(value['status'] == 'success' for value in danh_manifest_sample['papers'][danh_id_sample]['downloads'].values())
# Tính tay ở mục 15; fixture không phải số liệu corpus thật.
fixtures.CORPUS_SUMMARY


In [ ]:
if RUN_DANH_METRICS:
    danh_summary = summarize_corpus(deepcopy(fixtures.MANIFEST))
    assert danh_summary == fixtures.CORPUS_SUMMARY
    danh_benchmark = summarize_benchmark(deepcopy(fixtures.RESOURCE_SAMPLES), wall_seconds=20.0,
        final_disk_bytes=1000, environment=fixtures.BENCHMARK['environment'], cpu_only=True,
        cache_mode='cold', sampling_interval_seconds=10.0)
    assert danh_benchmark == fixtures.BENCHMARK
    print(danh_summary, danh_benchmark)
else:
    print('Danh: đang tắt kiểm hàm metrics thật. Thêm ca lỗi/resume theo mục 15.')

if RUN_INTEGRATION:
    # Chỉ bật khi cả bốn module thật xong. Chưa dùng phạm vi chính thức.
    danh_run = run_pipeline(deepcopy(fixtures.SCOPE), TEST_ROOT / 'integration',
                            policy=POLICY, resume=True)
    print(danh_run)
    # Mở file và đối chiếu theo mục 15; không coi in ra status là đủ nghiệm thu.


### Danh: kiểm tra tay và ghi bằng chứng

Đọc mục 15 của `docs/milestone1/PHAN_CONG_MILESTONE_1_4_NGUOI.md`.
Restart Kernel → chạy setup → các ô phần này là đủ, không cần fixture của phần khác.
Các assert trên chỉ kiểm một số bất biến; còn phải thêm ca lỗi/schema và mở đối chiếu nguồn/file.

| Ngày/môi trường | Hàm/case | Input/nguồn đối chiếu | Mong đợi | Thực tế/file | Kết luận/giới hạn |
|---|---|---|---|---|---|
| Chưa chạy thật | Điền sau triển khai | — | — | — | Chưa kiểm chứng |

Giữ mặc định cờ RUN=False trước commit, clear output. Không ghi pass cho ca chưa chạy.


## 5. Mẫu tình huống lỗi
Fixture giúp thống nhất ý nghĩa status, không chứng minh module đã xử lý lỗi đúng.
Khi viết thật cần kiểm tra PDF-only, gói hỏng/traversal, mã cũ, thiếu trang S2,
429/timeout, date thiếu, no_tex, input sai, resume và ghi file nguyên tử.


In [ ]:
# Output cuối kỳ vọng cho manifest giả lập, KHÔNG phải run thật.
expected_run = deepcopy(fixtures.RUN_RESULT)
assert expected_run['summary']['versions_success'] == 2
expected_run


In [ ]:
assert fixtures.DOWNLOAD_NO_TEX['status'] == 'no_tex'
assert fixtures.DOWNLOAD_FAILED['status'] == 'failed'
assert fixtures.DOWNLOAD_FAILED['source_bytes_after'] is None
assert fixtures.REFERENCE_FAILED['total_references'] is None
assert not fixtures.REFERENCE_FAILED['pagination_complete']
assert fixtures.DATE_RESULT_MISSING['dates']['1706.03762'] is None
print('Mẫu input/output nhất quán. Chưa chạy hay kiểm thử crawler thật.')


## 6. Hoàn thành đồ án sau triển khai

- Chạy đúng phạm vi thầy giao, ghi đủ mọi phiên bản kể cả không có TeX/lỗi.
- metadata.json và references.json đầy đủ, nguồn văn bản không bị đổi; giữ lệnh hình/bảng.
- Thống kê chất lượng dữ liệu và benchmark end-to-end Colab CPU, có cache mode và mẫu số.
- Báo cáo tiếng Anh trên Jupyter Book repo riêng; bảng/biểu đồ sinh lúc build từ dữ liệu lưu.
- README cài/chạy, requirements đã kiểm chứng, code công khai; không commit corpus hay key.
- Nộp link code + báo cáo qua Moodle đúng hạn. Giữ corpus cho Milestone 2–3.

Đọc thêm docs/milestone1/HUONG_DAN_THANH_VIEN_MILESTONE_1.md. Bộ khung này không thay việc triển khai,
benchmark, báo cáo và nộp bài. Không dùng fixture làm số liệu thật.
